# Module 1: Talk to the Machine

## Building Your First LLM-Powered Chat Application

In this module, you'll learn how to:
- Set up and configure an OpenAI-compatible client
- Make your first inference call to an LLM
- Build an interactive chat loop with conversation history
- Handle errors gracefully

---

## Step 1: Install Dependencies

First, let's install the required packages.

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

## Step 2: Import Libraries

We'll use:
- `openai` - The OpenAI Python SDK (works with any OpenAI-compatible API)
- `dotenv` - To load environment variables from a `.env` file
- `httpx` - HTTP client for custom configurations

In [ ]:
import os
import httpx
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

print("Libraries loaded successfully!")

## Step 3: Configure the OpenAI Client

We'll create an OpenAI client configured to use our API endpoint.

**Important:** Make sure your `.env` file contains:
```
OPENAI_BASE_URL=<your-api-endpoint>
OPENAI_API_KEY=<your-api-key>
```

In [ ]:
http_client = httpx.Client(verify=False)

# Initialize the OpenAI client
client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    http_client=http_client
)

print(f"Client configured!")
print(f"Base URL: {os.getenv('OPENAI_BASE_URL')}")

## Step 4: Make Your First Inference Call

Let's send a simple message to the LLM and get a response.

The `chat.completions.create()` method takes:
- `model` - The model name to use
- `messages` - A list of message objects with `role` and `content`
- `max_tokens` - Maximum tokens in the response

In [ ]:
MODEL_NAME = "gemma-4-26b-a4b-it"  

messages = [
    {"role": "user", "content": "Hello! What can you help me with today?"}
]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    max_tokens=500
)

assistant_message = response.choices[0].message.content
print("Assistant:", assistant_message)

## Step 5: Understanding the Response Object

Let's explore what the API returns.

In [ ]:
print("Full Response Object:")
print(f"  ID: {response.id}")
print(f"  Model: {response.model}")
print(f"  Created: {response.created}")
print(f"  Choices: {len(response.choices)}")
print()
print("First Choice:")
print(f"  Role: {response.choices[0].message.role}")
print(f"  Content: {response.choices[0].message.content[:100]}...")
print(f"  Finish Reason: {response.choices[0].finish_reason}")
print()
if response.usage:
    print("Token Usage:")
    print(f"  Prompt Tokens: {response.usage.prompt_tokens}")
    print(f"  Completion Tokens: {response.usage.completion_tokens}")
    print(f"  Total Tokens: {response.usage.total_tokens}")

## Step 6: Adding System Prompts

System prompts help set the behavior and personality of the assistant.

In [ ]:
messages = [
    {
        "role": "system", 
        "content": "You are a helpful coding assistant. You provide clear, concise explanations with code examples when relevant."
    },
    {
        "role": "user", 
        "content": "How do I read a JSON file in Python?"
    }
]

response = client.chat.completions.create(
    model="gemma-4-26b-a4b-it",
    messages=messages,
    max_tokens=2000
)

print("Content:", response.choices[0].message.content)

## Step 7: Building Conversation History

To have a multi-turn conversation, we need to maintain message history.

In [ ]:
class ChatSession:
    """A simple chat session that maintains conversation history."""
    
    def __init__(self, client, model_name, system_prompt=None):
        self.client = client
        self.model_name = model_name
        self.messages = []
        
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def chat(self, user_message):
        """Send a message and get a response."""
        self.messages.append({"role": "user", "content": user_message})
        
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=self.messages,
            max_tokens=1000
        )
        
        assistant_message = response.choices[0].message.content
        
        self.messages.append({"role": "assistant", "content": assistant_message})
        
        return assistant_message
    
    def get_history(self):
        """Return the conversation history."""
        return self.messages
    
    def clear_history(self):
        """Clear conversation history (keeps system prompt)."""
        self.messages = [msg for msg in self.messages if msg["role"] == "system"]

print("ChatSession class defined!")

In [ ]:
session = ChatSession(
    client=client,
    model_name=MODEL_NAME,
    system_prompt="You are a friendly AI assistant. Keep responses concise."
)

print("User: What is Python?")
response1 = session.chat("What is Python?")
print(f"Assistant: {response1}\n")

print("User: What are its main uses?")
response2 = session.chat("What are its main uses?")  # Model remembers context!
print(f"Assistant: {response2}\n")

print("User: Give me a simple example.")
response3 = session.chat("Give me a simple example.")  # Still has context!
print(f"Assistant: {response3}")

## Step 8: Interactive Chat Loop

Now let's build an interactive chat interface!

In [ ]:
def interactive_chat(client, model_name, system_prompt=None):
    """Run an interactive chat loop."""
    session = ChatSession(client, model_name, system_prompt)
    
    print("="*50)
    print("Interactive Chat Started!")
    print("Type 'quit' to exit, 'clear' to reset history")
    print("="*50)
    print()
    
    while True:
        try:
            user_input = input("You: ").strip()
            
            if user_input.lower() == 'quit':
                print("Goodbye!")
                break
            elif user_input.lower() == 'clear':
                session.clear_history()
                print("[History cleared]\n")
                continue
            elif not user_input:
                continue
            
            response = session.chat(user_input)
            print(f"\nAssistant: {response}\n")
            
        except KeyboardInterrupt:
            print("\nGoodbye!")
            break
        except Exception as e:
            print(f"\nError: {e}\n")

print("Interactive chat function defined!")
print("Run the next cell to start chatting.")

In [ ]:
interactive_chat(
    client=client,
    model_name=MODEL_NAME,
    system_prompt="You are a helpful AI assistant for the NAI MCP Workshop. Be concise and friendly."
)

---

## Summary

In this module, you learned:

1. **Setting up the client** - Configure OpenAI SDK with custom endpoints
2. **Making inference calls** - Use `chat.completions.create()` to get responses
3. **Understanding responses** - Parse the API response structure
4. **System prompts** - Set assistant behavior and personality
5. **Conversation history** - Maintain context across multiple turns
6. **Interactive chat** - Build a real chat interface

---

## Next Steps

Move to **Module 2: Extending the Brain** where you'll learn to build and deploy MCP servers to give your AI new capabilities!
